In [ ]:
import pandas as pd
import pdfplumber

In [13]:
def extract_tables_from_pdf(file_path, output_dir, target_pages):
    """Main orchestrator for opening the PDF and processing pages."""
    with pdfplumber.open(file_path) as pdf:
        for page_index, file_name in target_pages.items():
            page = pdf.pages[page_index]
            table = page.extract_table()
            
            if not table:
                print(f"Warning: No table found on page {page_index + 1}")
                continue

            # Convert to DataFrame (skipping header row for data)
            df = pd.DataFrame(table[1:], columns=table[0])

            # Clean column names and data
            df.columns = [str(c).replace('\n', ' ') if c else f"Unnamed_{i}" 
                          for i, c in enumerate(df.columns)]
            df = df.replace('\n', ' ', regex=True)
            
            # Delegate the saving logic to the new function
            save_table_to_csv(df, output_dir, file_name)

In [14]:
file_path = "data/original_data/participation_data/2024-sfia-topline.pdf"
output_dir = os.path.join("data", "extracted_data")
target_pages = {
    31: "SFIA_Aerobic_Activities.csv",
    32: "SFIA_Conditioning_Activities.csv",
    33: "SFIA_Team_Sports.csv"
}

extract_tables_from_pdf(file_path, output_dir, target_pages)

Skipping: SFIA_Aerobic_Activities.csv already exists in data\extracted_data
Skipping: SFIA_Conditioning_Activities.csv already exists in data\extracted_data
Skipping: SFIA_Team_Sports.csv already exists in data\extracted_data


In [8]:
def filter_by_code(main_table, mapping_table, target_cols_list, code_col_name="Code"):
    valid_codes = mapping_table[code_col_name].values

    mask = main_table[target_cols_list].isin(valid_codes).any(axis=1)

    return main_table[mask]

In [15]:
def bulk_filter_by_code(main_tables_list, mapping_table, target_cols_list, code_col_name="Code"):
    filtered_tables = []

    for table in main_tables_list:
        filtered_table = filter_by_code(table, mapping_table, target_cols_list, code_col_name)
        filtered_tables.append(filtered_table)
        
    return filtered_tables

In [9]:
main_table = pd.read_excel('data/original_data/injury_data/NEISS_2024.XLSX')
mapping_table = pd.read_csv('data/original_data/codes/NEISS_tables/product_codes.csv')

filtered_table = filter_by_code(main_table, mapping_table, ['Product_1', 'Product_2', 'Product_3'])

print(main_table.shape)
print(filtered_table.shape)

(361672, 25)
(98267, 25)


In [16]:
def replace_code_with_meaning(main_table, mapping_table, target_cols, meaning_col, code_col="Code"):
    # Ensure new object will be returned
    new_table = main_table.copy()
    mapping_dict = dict(zip(mapping_table[code_col], mapping_table[meaning_col]))

    # Apply the mapping to all target columns
    for col in target_cols:
        new_table[col] = new_table[col].map(mapping_dict).fillna(new_table[col])
    
    return new_table

In [17]:
def bulk_replace_codes_with_meanings(main_tables_list, mapping_table, target_cols, meaning_col, code_col="Code"):
    replaced_data_tables = []

    for table in main_tables_list:
        replaced_table = replace_code_with_meaning(table, mapping_table, target_cols, meaning_col, code_col)
        replaced_data_tables.append(replaced_table)

    return replaced_data_tables